# Huấn luyện Mô hình Chẩn đoán DR (Hybrid: CNN + SVD + GP)

Notebook này được thiết kế để chạy trên **Google Colab** hoặc **Kaggle Kernels** (miễn phí GPU).

## 1. Chuẩn bị môi trường & Dataset (APTOS 2019)
Dataset: [APTOS 2019 Blindness Detection](https://www.kaggle.com/c/aptos2019-blindness-detection/data)

In [ ]:
# Cài đặt thư viện cần thiết (chạy trên Colab/Kaggle)
!pip install tensorflow scikit-learn numpy opencv-python-headless matplotlib seaborn joblib

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.models import Model
import joblib

# Cấu hình
IMG_SIZE = 299  # InceptionV3 input size
BATCH_SIZE = 32
DATA_DIR = "../input/aptos2019-blindness-detection" # Đường dẫn trên Kaggle
# Nếu dùng Colab, bạn cần mount Drive hoặc tải data về

## 2. Tiền xử lý ảnh (CLAHE + Crop)

In [ ]:
def preprocess_image(image_path):
    """Đọc, crop viền đen, resize và áp dụng CLAHE"""
    img = cv2.imread(image_path)
    if img is None: return None
    
    # 1. Crop viền đen (tối giản)
    # (Code crop đầy đủ có thể phức tạp hơn, ở đây demo resize & CLAHE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    
    # 2. Chuyển sang LAB để áp dụng CLAHE lên kênh L
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    final_img = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)
    
    # 3. Chuẩn hóa cho InceptionV3
    final_img = preprocess_input(final_img) 
    return final_img

## 3. Trích xuất đặc trưng (CNN Feature Extraction)

In [ ]:
# Load Pre-trained InceptionV3 (bỏ lớp classification cuối)
base_model = InceptionV3(weights='imagenet', include_top=False, pooling='avg')
model_feature_extractor = Model(inputs=base_model.input, outputs=base_model.output)

# Freeze weighs (không train lại CNN, chỉ dùng để trích đặc trưng)
for layer in base_model.layers:
    layer.trainable = False

# Hàm trích xuất features cho toàn bộ dataset
def extract_features(df, img_dir):
    features = []
    labels = []
    
    for idx, row in df.iterrows():
        img_name = row['id_code'] + '.png'
        path = os.path.join(img_dir, img_name)
        
        img = preprocess_image(path)
        if img is not None:
            # Thêm 1 chiều batch (1, 299, 299, 3)
            img_batch = np.expand_dims(img, axis=0)
            feature = model_feature_extractor.predict(img_batch, verbose=0)
            
            features.append(feature.flatten())
            labels.append(row['diagnosis'])
            
        if idx % 100 == 0: print(f"Processed {idx} images")
            
    return np.array(features), np.array(labels)

# Giả sử bạn đã load dataframe train.csv
# df_train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
# X_features, y_labels = extract_features(df_train, os.path.join(DATA_DIR, 'train_images'))
# print("Feature shape:", X_features.shape)

## 4. Giảm chiều dữ liệu (SVD)

In [ ]:
# InceptionV3 ra vector 2048 chiều -> Giảm xuống thấp hơn (ví dụ 50-100) cho Gaussian Process
svd = TruncatedSVD(n_components=50, random_state=42)

# X_reduced = svd.fit_transform(X_features)
# print("Reduced shape:", X_reduced.shape)

## 5. Phân loại bằng Gaussian Process (GP)
GP giúp đưa ra độ bất định (uncertainty).

In [ ]:
kernel = 1.0 * RBF(1.0)
gpc = GaussianProcessClassifier(kernel=kernel, random_state=42, n_jobs=-1)

# gpc.fit(X_reduced, y_labels)
# print("Training finished!")

## 6. Lưu Model (Artifacts)
Lưu lại các file này để đưa vào thư mục `backend/models/` của project local.

In [ ]:
# 1. Lưu CNN (thực ra chỉ cần file weights hoặc load lại InceptionV3 từ keras)
# model_feature_extractor.save("cnn_feature_extractor.h5")

# 2. Lưu SVD
# joblib.dump(svd, "svd_reducer.pkl")

# 3. Lưu GP Classifier
# joblib.dump(gpc, "gp_classifier.pkl")